# Airbnb Listing Clustering

## Purpose
Run the selected K-Means pipeline and regenerate the two clustering CSV outputs. Exploratory alternatives, model-selection plots, and feature-ablation reasoning are documented in **`clustering rough work.ipynb`**.

## Selected candidate
- **Algorithm:** K-Means
- **Clusters:** `k = 2`
- **Rows:** all validated ML-ready listings
- **Feature preparation:** cap `minimum nights` at 365; apply `log1p` to `minimum nights`, `number of reviews`, and `calculated host listings count`
- **Feature selection:** exclude `service fee`, latitude, and longitude from the clustering matrix; retain neighbourhood-group indicators
- **Scaling:** `RobustScaler` for continuous features; binary/one-hot features remain 0/1
- **Reproducibility:** `random_state=42`, `n_init=10`

The current candidate was chosen because its silhouette score (~0.3912) was higher than the tested alternatives, it was highly stable across random seeds, and its two clusters had interpretable host-portfolio, minimum-stay, review-activity, and availability differences.

## Outputs
1. `data/ml/outputs/clustered_listings_kmeans_k2.csv` — original ML-ready rows plus the cluster label.
2. `data/ml/outputs/cluster_profile_kmeans_k2.csv` — compact numerical profile for each cluster.

Run this notebook from the project's `notebooks/` directory.

## 1. Import dependencies and load the validated ML-ready data

This cell reads the structured dataset and checks the assumptions the pipeline requires. It does not rerun EDA or modify the input file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler

# Paths are relative to notebooks/, keeping this notebook portable across machines.
DATA_PATH = Path("../data/ml/ml_ready_dataset.csv")
OUTPUT_DIR = Path("../data/ml/outputs")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input dataset not found: {DATA_PATH.resolve()}. "
        "Run this notebook from notebooks/ or update DATA_PATH."
    )

# Load the structured features exported by preprocessing.
df = pd.read_csv(DATA_PATH)

# Confirm required columns are available before feature engineering.
required_columns = [
    "price",
    "service fee",
    "minimum nights",
    "number of reviews",
    "calculated host listings count",
    "availability 365",
    "lat",
    "long"
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from ML-ready dataset: {missing_columns}")

# The current pipeline expects the validated, numeric, complete ML export.
if df.isna().any().any():
    raise ValueError("ML-ready input contains missing values; validate preprocessing first.")
if df.duplicated().any():
    raise ValueError("ML-ready input contains duplicate rows; validate preprocessing first.")
if len(df) == 0:
    raise ValueError("ML-ready input contains no listings.")

print(f"Loaded {len(df):,} listings and {df.shape[1]} input features.")

## 2. Prepare and scale the selected feature matrix

The feature engineering below is performed on a copy, preserving the original rows for export and interpretation. Exact coordinates are excluded because neighbourhood-group indicators already represent geographic categories in the model. `service fee` is excluded to reduce redundant pricing influence.

In [ ]:
# Preserve the source data unchanged; all model-specific transformations occur in this copy.
df_model = df.copy()

# Cap extreme minimum-stay requirements before applying the logarithmic transform.
# The 365-day cap is a documented project-level treatment for this learning dataset.
df_model["minimum nights"] = df_model["minimum nights"].clip(upper=365)

# Log-transform selected non-negative skewed features to reduce long-tail influence.
features_to_log = [
    "minimum nights",
    "number of reviews",
    "calculated host listings count"
]
for col in features_to_log:
    if (df_model[col] < 0).any():
        raise ValueError(f"Negative values found in {col}; resolve before log1p transformation.")
    df_model[col] = np.log1p(df_model[col])

# Exclude redundant service fee and exact coordinates from the clustering distance.
excluded_features = ["service fee", "lat", "long"]
X = df_model.drop(columns=excluded_features)

# Keep binary and one-hot encoded columns as 0/1; robust-scale continuous features.
binary_cols = [
    col for col in X.columns
    if X[col].nunique() <= 2
    and set(X[col].dropna().unique()).issubset({0, 1})
]
continuous_cols = [col for col in X.columns if col not in binary_cols]

scaler = RobustScaler()
X_scaled = X.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])

print(f"Model matrix: {X_scaled.shape[0]:,} listings × {X_scaled.shape[1]} features")
print(f"Continuous features scaled: {len(continuous_cols)}")
print(f"Binary/one-hot features preserved: {len(binary_cols)}")

## 3. Fit the selected K-Means model

Fit one reproducible two-cluster model on the selected scaled feature matrix. The resulting labels use the same row order as the input dataset.

In [ ]:
# Fit the selected candidate; seed and n_init keep the result reproducible.
model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)
labels = model.fit_predict(X_scaled)

# Store labels with the original row index to maintain row alignment.
cluster_labels = pd.Series(labels, index=df.index, name="cluster")

# Validate alignment and report the cluster distribution before export.
if len(cluster_labels) != len(df) or cluster_labels.isna().any():
    raise ValueError("Cluster assignments do not align with all input listings.")

cluster_counts = cluster_labels.value_counts().sort_index()
cluster_summary = cluster_counts.rename("listings").to_frame()
cluster_summary["share_pct"] = (
    cluster_summary["listings"] / len(df) * 100
).round(2)

display(cluster_summary)

## 4. Build the two output tables

The assignment file keeps the original ML-ready features and adds the cluster label. The profile file summarizes selected business-facing variables in original units, not scaled/log-transformed units. This makes the exports usable for downstream analysis without confusing transformed model values with the source features.

In [ ]:
# Prepare original-unit values for profiling; apply the same cap used in modeling.
df_profile = df.copy()
df_profile["minimum nights"] = df_profile["minimum nights"].clip(upper=365)
df_profile["cluster"] = cluster_labels.to_numpy()

# Create a compact business profile for each assigned cluster.
cluster_profile = df_profile.groupby("cluster").agg(
    listings=("price", "size"),
    price_mean=("price", "mean"),
    price_median=("price", "median"),
    minimum_nights_median=("minimum nights", "median"),
    reviews_mean=("number of reviews", "mean"),
    reviews_per_month_mean=("reviews per month", "mean"),
    host_listings_mean=("calculated host listings count", "mean"),
    host_listings_median=("calculated host listings count", "median"),
    availability_mean=("availability 365", "mean")
).round(2)

# Add each cluster's percentage of all listings.
cluster_profile["share_pct"] = (
    cluster_profile["listings"] / len(df_profile) * 100
).round(2)

# Preserve the source ML-ready features and append only the cluster assignment.
clustered_listings = df.copy()
clustered_listings["cluster"] = cluster_labels.to_numpy()

display(cluster_profile)

## 5. Export and validate the two CSV outputs

Outputs are written to `data/ml/outputs/`, separately from the input files. The input dataset is never overwritten.

In [ ]:
# Create the output directory if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep descriptive filenames so the model/version is explicit.
assignments_path = OUTPUT_DIR / "clustered_listings_kmeans_k2.csv"
profile_path = OUTPUT_DIR / "cluster_profile_kmeans_k2.csv"

# Export listing-level assignments and the compact cluster profile.
clustered_listings.to_csv(assignments_path, index=False)
cluster_profile.to_csv(profile_path, index=True)

# Final integrity checks ensure one cluster label exists for every input row.
assert len(clustered_listings) == len(df)
assert clustered_listings["cluster"].notna().all()
assert cluster_profile["listings"].sum() == len(df)

print("Cluster assignments saved to:", assignments_path.resolve())
print("Cluster profiles saved to:", profile_path.resolve())
print(f"Exported listings: {len(clustered_listings):,}")
print("\nFinal cluster distribution (%):")
display(
    (cluster_labels.value_counts(normalize=True).sort_index() * 100)
    .round(2)
    .rename("share_pct")
    .to_frame()
)